# SSVEP Performance Analysis

In [ ]:
import numpy as np
import pandas as pd
import mne
from mne_bids import BIDSPath, read_raw_bids
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import CCA
from scipy.linalg import eigh
from scipy.signal import cheby1, detrend, sosfiltfilt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Matplotlib settings
%matplotlib inline
plt.rcParams['font.family'] = 'Times New Roman'

TARGET_FREQS = [10.0, 12.0, 15.0]
ANALYSIS_TMIN = 0.5
ANALYSIS_TMAX = 4.5
MAX_ANALYSIS_FREQ = 100.0
SNR_NOISE_BW = 1.0
DECISION_WINDOW_S = ANALYSIS_TMAX - ANALYSIS_TMIN
FULL_TRIAL_CYCLE_S = 3.0 + 1.0 + 5.0 + 5.0


## 1. Load BIDS Data

In [ ]:
bids_root = 'dataset\\derivatives\\preprocessed'
subject = '03'
task = 'ssvep'
datatype = 'eeg'
suffix = 'eeg'

# Build BIDS path
bids_path = BIDSPath(subject=subject, task=task, suffix=suffix,
                     datatype=datatype, root=bids_root)

# Read data
raw = read_raw_bids(bids_path=bids_path)

# Load data into memory
raw.load_data()

# Print info
print(raw.info)
print(f"Sampling rate: {raw.info['sfreq']} Hz")
print(f"Channels: {len(raw.ch_names)}")

## 2. Event Processing

In [ ]:
# Extract events from BIDS annotations
all_events, all_event_dict = mne.events_from_annotations(raw)
event_dict = {str(name): code for name, code in all_event_dict.items()}
print('Detected event dict:', event_dict)

# Keep only stimulus events for SSVEP analysis
stim_event_id = {name: code for name, code in event_dict.items() if name.startswith('stim_')}
if not stim_event_id:
    raise RuntimeError('No stim_* events detected. Check events.tsv trial_type.')

# Main analysis window: 0.5-4.5s post-stimulus
epochs_stim_full = mne.Epochs(
    raw,
    all_events,
    event_id=stim_event_id,
    tmin=ANALYSIS_TMIN,
    tmax=ANALYSIS_TMAX,
    baseline=None,
    preload=True,
)

# Occipital/parietal-occipital channels for SSVEP classification and SNR
occipital_chs = ['PO7', 'PO3', 'POz', 'PO4', 'PO8', 'O1', 'Oz', 'O2', 'Pz', 'P3', 'P4']
available_chs = [ch for ch in occipital_chs if ch in epochs_stim_full.ch_names]
if not available_chs:
    raise RuntimeError(f'No occipital channels found. Available: {epochs_stim_full.ch_names}')

epochs = epochs_stim_full.copy().pick_channels(available_chs)
print(f'Main analysis window: {ANALYSIS_TMIN:.1f}-{ANALYSIS_TMAX:.1f}s')
print(f'Stimulus trials: {len(epochs)}')
print(f'Channels for classification/SNR: {available_chs}')

# Rest-eyes-open PSD for baseline normalization (skip if unavailable)
rest_psds = None
rest_freqs = None
rest_event_id = {name: code for name, code in event_dict.items() if name == 'rest_eyes_open'}
if rest_event_id:
    rest_epochs = mne.Epochs(
        raw,
        all_events,
        event_id=rest_event_id,
        tmin=0,
        tmax=120,
        baseline=None,
        preload=True,
    )
    rest_epochs.pick_channels(available_chs)
    rest_psd_obj = rest_epochs.compute_psd(
        method='welch',
        fmin=5,
        fmax=MAX_ANALYSIS_FREQ,
        n_fft=512,
        n_overlap=256,
    )
    rest_psds, rest_freqs = rest_psd_obj.get_data(return_freqs=True)
    rest_psds = rest_psds.mean(axis=0)  # shape: channels x frequencies
    print('Rest-eyes-open PSD computed for baseline normalization.')
else:
    print('rest_eyes_open not found; skipping baseline normalization.')


## 3. Power Spectral Density (PSD) Analysis

In [ ]:

spectrum = epochs.compute_psd(method='welch', fmin=5, fmax=MAX_ANALYSIS_FREQ, n_fft=512, n_overlap=256)
psds, freqs = spectrum.get_data(return_freqs=True)
avg_psd = np.mean(psds, axis=(0, 1))

fig, ax = plt.subplots(figsize=(10, 5))
y_data = 10 * np.log10(avg_psd)

ax.plot(freqs, y_data)
ax.set_xlim(0, 50)

for f in TARGET_FREQS:
    ax.axvline(f, color='r', linestyle='--', alpha=0.5)
for f in [20, 24, 30]:
    ax.axvline(f, color='g', linestyle='--', alpha=0.5)
plt.title(f'Overall Stimulus PSD ({ANALYSIS_TMIN:.1f}-{ANALYSIS_TMAX:.1f}s)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (dB)')
ax.grid(True, which='major', axis='both')

ax_inset = inset_axes(ax, width="25%", height="25%", loc='upper right')
ax_inset.plot(freqs, y_data, alpha=0.5)
ax_inset.set_xlim(0, 100)
ax_inset.set_title('Full (0-100 Hz)')
ax_inset.tick_params(axis='both', which='both', labelleft=False)
ax_inset.grid(True, linestyle=':', alpha=0.5)

plt.show()

## 4. SNR Calculation Functions

In [ ]:
def parse_stim_label(event_name):
    """Parse BIDS trial_type label, e.g. stim_10Hz_trans50_white."""
    event_name = str(event_name)
    parts = event_name.split('_')
    if len(parts) != 4 or parts[0] != 'stim':
        raise ValueError(f'Invalid stim label: {event_name}')
    return {
        'Event': event_name,
        'Frequency': float(parts[1].replace('Hz', '')),
        'Transparency': int(parts[2].replace('trans', '')),
        'Color': parts[3],
    }


def valid_harmonics(target_freq, max_freq=MAX_ANALYSIS_FREQ, max_harmonics=3):
    """Return valid target harmonics within the lowpass range."""
    return [target_freq * h for h in range(1, max_harmonics + 1) if target_freq * h <= max_freq + 1e-9]


def calculate_harmonic_snr(psd_data, freqs, target_freq, noise_bw=SNR_NOISE_BW,
                           max_freq=MAX_ANALYSIS_FREQ, max_harmonics=3):
    """
    Compute SNR for fundamental and available harmonics.
    Returns linear and dB SNR per channel; scalar for 1-D input.
    """
    psd_array = np.asarray(psd_data)
    was_1d = psd_array.ndim == 1
    psd_2d = np.atleast_2d(psd_array)
    freqs = np.asarray(freqs)
    bin_width = np.nanmedian(np.diff(freqs))
    guard_hz = max(0.25, 1.5 * bin_width)

    harmonic_ratios = []
    for harmonic_freq in valid_harmonics(target_freq, max_freq=max_freq, max_harmonics=max_harmonics):
        idx_target = int(np.argmin(np.abs(freqs - harmonic_freq)))
        noise_mask = (np.abs(freqs - harmonic_freq) <= noise_bw) & (np.abs(freqs - harmonic_freq) > guard_hz)
        if not np.any(noise_mask):
            continue
        target_pow = psd_2d[:, idx_target]
        noise_pow = np.nanmean(psd_2d[:, noise_mask], axis=1)
        harmonic_ratios.append(target_pow / noise_pow)

    if not harmonic_ratios:
        snr_linear = np.full(psd_2d.shape[0], np.nan)
    else:
        snr_linear = np.nanmean(np.vstack(harmonic_ratios), axis=0)
    snr_db = 10 * np.log10(snr_linear)

    if was_1d:
        return float(snr_linear[0]), float(snr_db[0])
    return snr_linear, snr_db


def calculate_baseline_power_db(trial_psds, freqs, rest_psds, rest_freqs, target_freq,
                                max_freq=MAX_ANALYSIS_FREQ, max_harmonics=3):
    """Compute dB gain of stimulus power relative to rest-eyes-open baseline."""
    if rest_psds is None or rest_freqs is None:
        return np.nan

    harmonic_freqs = valid_harmonics(target_freq, max_freq=max_freq, max_harmonics=max_harmonics)
    if not harmonic_freqs:
        return np.nan

    target_indices = [int(np.argmin(np.abs(freqs - f))) for f in harmonic_freqs]
    rest_interp = np.vstack([np.interp(freqs, rest_freqs, ch_psd) for ch_psd in rest_psds])
    n_ch = min(trial_psds.shape[0], rest_interp.shape[0])

    stim_power = np.nanmean(trial_psds[:n_ch, :][:, target_indices], axis=1)
    rest_power = np.nanmean(rest_interp[:n_ch, :][:, target_indices], axis=1)
    return float(np.nanmean(10 * np.log10(stim_power / rest_power)))


## 5. Factor Analysis: Transparency & Color Effects on Signal Quality

In [ ]:
snr_rows = []

spectrum = epochs.compute_psd(
    method='welch',
    fmin=5,
    fmax=MAX_ANALYSIS_FREQ,
    n_fft=512,
    n_overlap=256,
)
psds, freqs = spectrum.get_data(return_freqs=True)
id_to_name = {code: name for name, code in epochs.event_id.items()}

for i, etype in enumerate(epochs.events[:, 2]):
    label = id_to_name[int(etype)]
    info = parse_stim_label(label)
    trial_psds = psds[i]
    snr_linear_ch, snr_db_ch = calculate_harmonic_snr(trial_psds, freqs, info['Frequency'])

    snr_rows.append({
        'Trial_Index': i,
        **info,
        'SNR_linear': float(np.nanmean(snr_linear_ch)),
        'SNR_dB': float(np.nanmean(snr_db_ch)),
        'Rest_Normalized_Power_dB': calculate_baseline_power_db(
            trial_psds, freqs, rest_psds, rest_freqs, info['Frequency']
        ),
        'Harmonics_Used': ','.join(f'{f:g}' for f in valid_harmonics(info['Frequency'])),
    })

df_results = pd.DataFrame(snr_rows)
print(f'Successfully processed {len(df_results)} stimulus trials.')
display(df_results.head())

condition_summary = (
    df_results
    .groupby(['Frequency', 'Transparency', 'Color'])
    .agg(
        Trials=('SNR_dB', 'size'),
        Mean_SNR_dB=('SNR_dB', 'mean'),
        SD_SNR_dB=('SNR_dB', 'std'),
        Mean_Rest_Norm_dB=('Rest_Normalized_Power_dB', 'mean'),
    )
    .reset_index()
)
display(condition_summary)


### 5.1 Visualization of Factor Effects

In [ ]:
if not df_results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    sns.boxplot(data=df_results, x='Transparency', y='SNR_dB', hue='Transparency', palette='Set2', ax=axes[0],
                legend=False)
    sns.stripplot(data=df_results, x='Transparency', y='SNR_dB', color='black', alpha=0.3, ax=axes[0])
    axes[0].set_title('Impact of Transparency on Harmonic SNR')
    axes[0].set_ylabel('SNR (dB)')

    sns.boxplot(data=df_results, x='Color', y='SNR_dB', hue='Color', palette='Set3', ax=axes[1], legend=False)
    sns.stripplot(data=df_results, x='Color', y='SNR_dB', color='black', alpha=0.3, ax=axes[1])
    axes[1].set_title('Impact of Color Mode on Harmonic SNR')
    axes[1].set_ylabel('SNR (dB)')

    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_results, x='Frequency', y='SNR_dB', hue='Transparency', errorbar='se')
    plt.title('SNR across Frequencies and Transparency Levels')
    plt.ylabel('SNR (dB)')
    plt.show()

    if df_results['Rest_Normalized_Power_dB'].notna().any():
        plt.figure(figsize=(10, 5))
        sns.barplot(data=df_results, x='Frequency', y='Rest_Normalized_Power_dB', hue='Color', errorbar='se')
        plt.title('Stimulus Power Gain vs Eyes-open Rest')
        plt.ylabel('Rest-normalized power (dB)')
        plt.show()

    anova_data = df_results.dropna(subset=['SNR_dB']).copy()
    if len(anova_data) >= 12:
        model = smf.ols('SNR_dB ~ C(Frequency) * C(Transparency) * C(Color)', data=anova_data).fit()
        print('SNR multi-factor ANOVA (single-subject exploratory)')
        display(sm.stats.anova_lm(model, typ=2))


## 6. ITPC Phase Locking Analysis

In [ ]:
def calculate_fft_itpc(data, sfreq, target_freq):
    """FFT-bin ITPC across trials, averaged across channels."""
    data = np.asarray(data)
    if data.shape[0] < 2:
        return np.nan
    fft_vals = np.fft.rfft(data, axis=-1)
    fft_freqs = np.fft.rfftfreq(data.shape[-1], d=1.0 / sfreq)
    idx = int(np.argmin(np.abs(fft_freqs - target_freq)))
    phase = np.angle(fft_vals[:, :, idx])
    itpc_ch = np.abs(np.mean(np.exp(1j * phase), axis=0))
    return float(np.nanmean(itpc_ch))


itpc_rows = []
sfreq = epochs.info['sfreq']
for label in epochs.event_id.keys():
    info = parse_stim_label(label)
    data = epochs[label].get_data()
    harmonic_itpcs = []

    for harmonic_order, harmonic_freq in enumerate(valid_harmonics(info['Frequency']), start=1):
        itpc_value = calculate_fft_itpc(data, sfreq, harmonic_freq)
        harmonic_itpcs.append(itpc_value)
        itpc_rows.append({
            **info,
            'Trials': data.shape[0],
            'Harmonic_Order': harmonic_order,
            'Harmonic_Freq': harmonic_freq,
            'ITPC': itpc_value,
        })

    itpc_rows.append({
        **info,
        'Trials': data.shape[0],
        'Harmonic_Order': 0,
        'Harmonic_Freq': np.nan,
        'ITPC': float(np.nanmean(harmonic_itpcs)),
    })

df_itpc = pd.DataFrame(itpc_rows)
print('ITPC summary; Harmonic_Order=0 = mean ITPC across available harmonics for that condition.')
display(df_itpc.sort_values(['Frequency', 'Transparency', 'Color', 'Harmonic_Order']))

plt.figure(figsize=(10, 5))
sns.barplot(
    data=df_itpc[df_itpc['Harmonic_Order'] > 0],
    x='Frequency',
    y='ITPC',
    hue='Harmonic_Order',
    errorbar=None,
)
plt.title('ITPC by Fundamental and Harmonics')
plt.ylim(0, 1.05)
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(
    data=df_itpc[df_itpc['Harmonic_Order'] == 0],
    x='Transparency',
    y='ITPC',
    hue='Color',
    errorbar=None,
)
plt.title('Mean Harmonic ITPC by Transparency and Color')
plt.ylim(0, 1.05)
plt.show()


## 7. Topographic Map: Occipital Energy Distribution

In [ ]:
# Plot target-frequency power topomap (not time-domain evoked amplitude)

def has_valid_position(ch_info):
    loc = ch_info['loc'][:3]
    return np.isfinite(loc).all() and np.linalg.norm(loc) > 0


freq_conditions = TARGET_FREQS
topo_epochs = epochs_stim_full.copy()
valid_topo_chs = [ch['ch_name'] for ch in topo_epochs.info['chs'] if has_valid_position(ch)]

if len(valid_topo_chs) < 4:
    print('Fewer than 4 valid electrode positions; skipping topomap. Check electrodes.tsv / montage.')
else:
    topo_epochs.pick_channels(valid_topo_chs)
    fig, axes = plt.subplots(1, len(freq_conditions), figsize=(5 * len(freq_conditions), 4))
    axes = np.atleast_1d(axes)

    for ax, freq in zip(axes, freq_conditions):
        labels = []
        for label in topo_epochs.event_id.keys():
            try:
                if parse_stim_label(label)['Frequency'] == freq:
                    labels.append(label)
            except ValueError:
                pass

        if not labels:
            ax.set_title(f'{freq:g} Hz: no trials')
            ax.axis('off')
            continue

        topo_spectrum = topo_epochs[labels].compute_psd(
            method='welch',
            fmin=max(5, freq - 1),
            fmax=min(MAX_ANALYSIS_FREQ, freq + 1),
            n_fft=512,
            n_overlap=256,
        )
        topo_psds, topo_freqs = topo_spectrum.get_data(return_freqs=True)
        idx_freq = int(np.argmin(np.abs(topo_freqs - freq)))
        ch_power_db = 10 * np.log10(np.nanmean(topo_psds[:, :, idx_freq], axis=0) * 1e12)

        im, _ = mne.viz.plot_topomap(
            ch_power_db,
            topo_epochs.info,
            axes=ax,
            show=False,
            contours=0,
            cmap='viridis',
        )
        ax.set_title(f'{freq:g} Hz target power')

    fig.colorbar(im, ax=axes.tolist(), shrink=0.75, label='Power (dB uV²/Hz)')
    plt.show()


def channel_harmonic_snr_db(psd_trials, freqs, target_freq, noise_bw=SNR_NOISE_BW,
                            max_freq=MAX_ANALYSIS_FREQ, max_harmonics=3):
    """Return channel-wise mean harmonic SNR in dB for data shaped trials x channels x freqs."""
    freqs = np.asarray(freqs)
    bin_width = np.nanmedian(np.diff(freqs))
    guard_hz = max(0.25, 1.5 * bin_width)
    harmonic_snrs = []

    for harmonic_freq in valid_harmonics(target_freq, max_freq=max_freq, max_harmonics=max_harmonics):
        idx_target = int(np.argmin(np.abs(freqs - harmonic_freq)))
        noise_mask = (
                (np.abs(freqs - harmonic_freq) <= noise_bw)
                & (np.abs(freqs - harmonic_freq) > guard_hz)
        )
        if not np.any(noise_mask):
            continue
        target_power = psd_trials[:, :, idx_target]
        noise_power = np.nanmean(psd_trials[:, :, noise_mask], axis=2)
        valid = (target_power > 0) & (noise_power > 0)
        snr_db = np.full(target_power.shape, np.nan)
        snr_db[valid] = 10 * np.log10(target_power[valid] / noise_power[valid])
        harmonic_snrs.append(snr_db)

    if not harmonic_snrs:
        return np.full(psd_trials.shape[1], np.nan)
    return np.nanmean(np.stack(harmonic_snrs, axis=0), axis=(0, 1))


if len(valid_topo_chs) >= 4:
    fig, axes = plt.subplots(1, len(freq_conditions), figsize=(5 * len(freq_conditions), 4))
    axes = np.atleast_1d(axes)
    im = None

    for ax, freq in zip(axes, freq_conditions):
        labels = []
        for label in topo_epochs.event_id.keys():
            try:
                if parse_stim_label(label)['Frequency'] == freq:
                    labels.append(label)
            except ValueError:
                pass

        if not labels:
            ax.set_title(f'{freq:g} Hz: no trials')
            ax.axis('off')
            continue

        harmonic_freqs = valid_harmonics(freq)
        fmin = max(5, min(harmonic_freqs) - SNR_NOISE_BW)
        fmax = min(MAX_ANALYSIS_FREQ, max(harmonic_freqs) + SNR_NOISE_BW)
        topo_spectrum = topo_epochs[labels].compute_psd(
            method='welch',
            fmin=fmin,
            fmax=fmax,
            n_fft=512,
            n_overlap=256,
        )
        topo_psds, topo_freqs = topo_spectrum.get_data(return_freqs=True)
        ch_snr_db = channel_harmonic_snr_db(topo_psds, topo_freqs, freq)

        im, _ = mne.viz.plot_topomap(
            ch_snr_db,
            topo_epochs.info,
            axes=ax,
            show=False,
            contours=0,
            cmap='viridis',
        )
        ax.set_title(f'{freq:g} Hz harmonic SNR')

    if im is not None:
        fig.colorbar(im, ax=axes.tolist(), shrink=0.75, label='Mean harmonic SNR (dB)')
    fig.suptitle('Frequency-level SSVEP topoplot: fundamental + harmonics SNR', y=1.02)
    plt.show()

if len(valid_topo_chs) >= 4:
    condition_labels = []
    for label in topo_epochs.event_id.keys():
        try:
            info = parse_stim_label(label)
            condition_labels.append((info['Frequency'], info['Transparency'], info['Color'], label))
        except ValueError:
            pass
    condition_labels = sorted(condition_labels)

    n_cols = 4
    n_rows = int(np.ceil(len(condition_labels) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    im = None

    for ax, (freq, trans, color, label) in zip(axes, condition_labels):
        harmonic_freqs = valid_harmonics(freq)
        fmin = max(5, min(harmonic_freqs) - SNR_NOISE_BW)
        fmax = min(MAX_ANALYSIS_FREQ, max(harmonic_freqs) + SNR_NOISE_BW)
        topo_spectrum = topo_epochs[label].compute_psd(
            method='welch',
            fmin=fmin,
            fmax=fmax,
            n_fft=512,
            n_overlap=256,
        )
        topo_psds, topo_freqs = topo_spectrum.get_data(return_freqs=True)
        ch_snr_db = channel_harmonic_snr_db(topo_psds, topo_freqs, freq)

        im, _ = mne.viz.plot_topomap(
            ch_snr_db,
            topo_epochs.info,
            axes=ax,
            show=False,
            contours=0,
            cmap='viridis',
        )
        ax.set_title(f'{freq:g}Hz T{trans} {color}')

    for ax in axes[len(condition_labels):]:
        ax.axis('off')

    if im is not None:
        fig.colorbar(im, ax=axes.tolist(), shrink=0.7, label='Mean harmonic SNR (dB)')
    fig.suptitle('Condition-level SSVEP topoplot: fundamental + harmonics SNR', y=1.002)
    plt.show()



## 8. Canonical Correlation Analysis (CCA)

In [ ]:
def get_reference_signals(freq, sfreq, n_samples, max_freq=MAX_ANALYSIS_FREQ, max_harmonics=3):
    """Generate sine/cosine reference signals within the lowpass range."""
    t = np.arange(n_samples) / sfreq
    ref = []
    for harmonic_freq in valid_harmonics(freq, max_freq=max_freq, max_harmonics=max_harmonics):
        ref.append(np.sin(2 * np.pi * harmonic_freq * t))
        ref.append(np.cos(2 * np.pi * harmonic_freq * t))
    return np.array(ref).T


def prepare_cca_data(data):
    """Detrend and standardize to samples x channels."""
    x = detrend(data, axis=1).T
    x = x - np.mean(x, axis=0, keepdims=True)
    scale = np.std(x, axis=0, keepdims=True)
    scale[scale == 0] = 1.0
    return x / scale


def cca_analysis(data, freqs, sfreq, max_freq=MAX_ANALYSIS_FREQ, max_harmonics=3):
    """Return CCA correlation coefficients for each frequency."""
    n_samples = data.shape[1]
    x = prepare_cca_data(data)
    scores = {}

    for freq in freqs:
        ref = get_reference_signals(freq, sfreq, n_samples, max_freq=max_freq, max_harmonics=max_harmonics)
        cca = CCA(n_components=1, max_iter=1000)
        cca.fit(x, ref)
        x_score, y_score = cca.transform(x, ref)
        corr = np.corrcoef(x_score[:, 0], y_score[:, 0])[0, 1]
        scores[freq] = corr if np.isfinite(corr) else -np.inf
    return scores


def score_margin(scores):
    ordered = sorted(scores.values(), reverse=True)
    if len(ordered) < 2:
        return np.nan
    return float(ordered[0] - ordered[1])


def summarize_accuracy(df, group_cols=None, acc_col='Correct'):
    """Accuracy summary with Wilson 95% CI."""
    if group_cols is None:
        group_cols = []
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    if group_cols:
        grouped = df.groupby(group_cols)[acc_col].agg(Correct='sum', Trials='count', Accuracy='mean').reset_index()
    else:
        grouped = pd.DataFrame({
            'Correct': [df[acc_col].sum()],
            'Trials': [df[acc_col].count()],
            'Accuracy': [df[acc_col].mean()],
        })

    ci = grouped.apply(
        lambda row: proportion_confint(int(row['Correct']), int(row['Trials']), alpha=0.05, method='wilson'),
        axis=1,
    )
    grouped['CI95_Low'] = [x[0] for x in ci]
    grouped['CI95_High'] = [x[1] for x in ci]
    return grouped


def evaluate_epochs(epochs_obj, target_freqs, method='CCA'):
    """Evaluate frequency recognition performance per epoch."""
    method = method.upper()
    sfreq = epochs_obj.info['sfreq']
    id_to_name = {code: name for name, code in epochs_obj.event_id.items()}
    data_all = epochs_obj.get_data()
    rows = []

    for i, event_code in enumerate(epochs_obj.events[:, 2]):
        label = id_to_name[int(event_code)]
        info = parse_stim_label(label)
        data = data_all[i]

        if method == 'CCA':
            scores = cca_analysis(data, target_freqs, sfreq)
        elif method == 'FBCCA':
            scores = fbcca_analysis(data, target_freqs, sfreq)
        else:
            raise ValueError(f'Unknown method: {method}')

        pred_freq = max(scores, key=scores.get)
        rows.append({
            'Trial_Index': i,
            **info,
            'True_Freq': info['Frequency'],
            'Pred_Freq': pred_freq,
            'Correct': int(pred_freq == info['Frequency']),
            'Max_Score': float(scores[pred_freq]),
            'Score_Margin': score_margin(scores),
            'Method': method,
            'Decision_Window_s': epochs_obj.times[-1] - epochs_obj.times[0],
        })

    return pd.DataFrame(rows)


### 8.1 Evaluate CCA Performance


In [ ]:
print('Processing CCA...')
df_cca = evaluate_epochs(epochs, TARGET_FREQS, method='CCA')

print(f"Overall CCA accuracy: {df_cca['Correct'].mean():.2%}")
print('\n--- CCA accuracy by transparency ---')
display(summarize_accuracy(df_cca, 'Transparency'))

print('\n--- CCA accuracy by color ---')
display(summarize_accuracy(df_cca, 'Color'))

print('\n--- CCA accuracy: freq x transparency x color ---')
display(summarize_accuracy(df_cca, ['Frequency', 'Transparency', 'Color']))

print('\n--- CCA confusion matrix ---')
display(pd.crosstab(df_cca['True_Freq'], df_cca['Pred_Freq'], rownames=['True'], colnames=['Pred']))

plt.figure(figsize=(10, 6))
sns.boxplot(x='Transparency', y='Score_Margin', hue='Color', data=df_cca)
plt.title('CCA Score Margin across Conditions')
plt.ylabel('Top1 - Top2 CCA score')
plt.show()


## 9. FBCCA & TRCA Algorithms


In [ ]:
FBCCA_LOW_CUTS = [6.0, 14.0, 22.0, 30.0]


def filter_bank(data, sfreq, band_idx, max_freq=MAX_ANALYSIS_FREQ):
    """Create filter bank sub-bands within 40 Hz for FBCCA."""
    nyq = sfreq / 2
    low = FBCCA_LOW_CUTS[band_idx]
    high = min(max_freq, nyq - 1.0)
    if low >= high:
        raise ValueError(f'Invalid sub-band: low={low}, high={high}')
    sos = cheby1(4, 1, [low / nyq, high / nyq], btype='bandpass', output='sos')
    return sosfiltfilt(sos, data, axis=-1)


def fbcca_analysis(data, freqs, sfreq, n_bands=4):
    """FBCCA recognition with sub-bands and harmonics limited to MAX_ANALYSIS_FREQ."""
    n_bands = min(n_bands, len(FBCCA_LOW_CUTS))
    n_samples = data.shape[1]
    rho = np.zeros((n_bands, len(freqs)))
    weights = np.array([np.power(i + 1, -1.25) + 0.25 for i in range(n_bands)])

    for band_idx in range(n_bands):
        filtered_data = filter_bank(data, sfreq, band_idx)
        x = prepare_cca_data(filtered_data)
        for freq_idx, freq in enumerate(freqs):
            ref = get_reference_signals(freq, sfreq, n_samples)
            cca = CCA(n_components=1, max_iter=1000)
            cca.fit(x, ref)
            x_score, y_score = cca.transform(x, ref)
            corr = np.corrcoef(x_score[:, 0], y_score[:, 0])[0, 1]
            rho[band_idx, freq_idx] = corr if np.isfinite(corr) else 0.0

    final_scores = np.dot(weights, np.square(rho))
    return {freqs[i]: final_scores[i] for i in range(len(freqs))}


def demean_trials(data):
    return data - data.mean(axis=-1, keepdims=True)


def train_trca_filters(train_data, train_labels, target_freqs):
    """Train standard TRCA spatial filters and frequency templates."""
    filters = {}
    templates = {}
    n_channels = train_data.shape[1]

    for freq in target_freqs:
        x_class = demean_trials(train_data[train_labels == freq])
        if len(x_class) < 2:
            continue

        s_mat = np.zeros((n_channels, n_channels))
        q_mat = np.zeros((n_channels, n_channels))
        for i in range(len(x_class)):
            xi = x_class[i]
            q_mat += xi @ xi.T
            for j in range(i + 1, len(x_class)):
                xj = x_class[j]
                s_mat += xi @ xj.T + xj @ xi.T

        reg = 1e-6 * np.trace(q_mat) / n_channels if np.trace(q_mat) > 0 else 1e-6
        eigvals, eigvecs = eigh(s_mat, q_mat + reg * np.eye(n_channels))
        w = eigvecs[:, np.argmax(eigvals)]
        filters[freq] = w / (np.linalg.norm(w) + 1e-12)
        templates[freq] = x_class.mean(axis=0)

    return filters, templates


def trca_classify(data, filters, templates):
    """Classify frequency using TRCA spatial filters and templates."""
    x = demean_trials(np.asarray(data))
    scores = {}
    for freq, w in filters.items():
        test_proj = w @ x
        template_proj = w @ templates[freq]
        corr = np.corrcoef(test_proj, template_proj)[0, 1]
        scores[freq] = corr if np.isfinite(corr) else -np.inf
    return scores


def evaluate_trca_loo(epochs_obj, target_freqs):
    """Leave-one-out TRCA to prevent test trial from influencing its own template."""
    data_all = epochs_obj.get_data()
    id_to_name = {code: name for name, code in epochs_obj.event_id.items()}
    labels = []
    meta = []
    for event_code in epochs_obj.events[:, 2]:
        info = parse_stim_label(id_to_name[int(event_code)])
        labels.append(info['Frequency'])
        meta.append(info)
    labels = np.array(labels, dtype=float)

    rows = []
    for i, info in enumerate(meta):
        train_mask = np.ones(len(data_all), dtype=bool)
        train_mask[i] = False
        filters, templates = train_trca_filters(data_all[train_mask], labels[train_mask], target_freqs)
        scores = trca_classify(data_all[i], filters, templates)
        pred_freq = max(scores, key=scores.get)
        rows.append({
            'Trial_Index': i,
            **info,
            'True_Freq': info['Frequency'],
            'Pred_Freq': pred_freq,
            'Correct': int(pred_freq == info['Frequency']),
            'Max_Score': float(scores[pred_freq]),
            'Score_Margin': score_margin(scores),
            'Method': 'TRCA_LOO',
            'Decision_Window_s': epochs_obj.times[-1] - epochs_obj.times[0],
        })
    return pd.DataFrame(rows)


## 10. CCA vs FBCCA vs TRCA Performance


In [ ]:
print('Comparing CCA, FBCCA, and TRCA...')
df_fbcca = evaluate_epochs(epochs, TARGET_FREQS, method='FBCCA')
df_trca = evaluate_trca_loo(epochs, TARGET_FREQS)

base_cols = ['Trial_Index', 'Event', 'True_Freq', 'Frequency', 'Transparency', 'Color']
df_comp = df_cca[base_cols].copy()

for name, df_method in [('CCA', df_cca), ('FBCCA', df_fbcca), ('TRCA', df_trca)]:
    method_cols = df_method[base_cols + ['Pred_Freq', 'Correct', 'Max_Score', 'Score_Margin']].rename(columns={
        'Pred_Freq': f'Pred_{name}',
        'Correct': f'Acc_{name}',
        'Max_Score': f'Score_{name}',
        'Score_Margin': f'Margin_{name}',
    })
    df_comp = df_comp.merge(method_cols, on=base_cols, how='left')

display(df_comp.head())


## 11. ITR & Short-Window Analysis


In [ ]:
def calculate_itr(n_targets, accuracy, decision_seconds):
    """Information Transfer Rate, bits/min。"""
    accuracy = float(np.clip(accuracy, 1e-12, 1 - 1e-12))
    if accuracy <= 1 / n_targets:
        return 0.0
    bits = (
            np.log2(n_targets)
            + accuracy * np.log2(accuracy)
            + (1 - accuracy) * np.log2((1 - accuracy) / (n_targets - 1))
    )
    return float(bits * 60 / decision_seconds)


algorithms = ['CCA', 'FBCCA', 'TRCA']
summary_rows = []
for algo in algorithms:
    acc = df_comp[f'Acc_{algo}'].mean()
    summary_rows.append({
        'Algorithm': algo,
        'Accuracy': acc,
        'ITR_WindowOnly_bits_min': calculate_itr(len(TARGET_FREQS), acc, DECISION_WINDOW_S),
        'ITR_FullTrialCycle_bits_min': calculate_itr(len(TARGET_FREQS), acc, FULL_TRIAL_CYCLE_S),
        'Mean_Score_Margin': df_comp[f'Margin_{algo}'].mean(),
    })

print('=== Algorithm Performance Overview ===')
df_algorithm_summary = pd.DataFrame(summary_rows)
display(df_algorithm_summary)

for algo in algorithms:
    print(f'\n=== {algo}: Performance by transparency ===')
    display(summarize_accuracy(df_comp, 'Transparency', acc_col=f'Acc_{algo}'))

    print(f'\n=== {algo}: Performance by color ===')
    display(summarize_accuracy(df_comp, 'Color', acc_col=f'Acc_{algo}'))

    print(f'\n=== {algo}: Performance: freq x transparency x color ===')
    display(summarize_accuracy(df_comp, ['Frequency', 'Transparency', 'Color'], acc_col=f'Acc_{algo}'))

    print(f'\n=== {algo}: Confusion matrix ===')
    display(pd.crosstab(df_comp['True_Freq'], df_comp[f'Pred_{algo}'], rownames=['True'], colnames=[f'Pred {algo}']))

# Visualize algorithm performance by transparency/color
plot_rows = []
for algo in algorithms:
    tmp = df_comp[['Transparency', 'Color', f'Acc_{algo}']].rename(columns={f'Acc_{algo}': 'Accuracy'})
    tmp['Algorithm'] = algo
    plot_rows.append(tmp)
df_plot = pd.concat(plot_rows, ignore_index=True)

plt.figure(figsize=(10, 5))
sns.barplot(x='Transparency', y='Accuracy', hue='Algorithm', data=df_plot, errorbar='se')
plt.title('Algorithm Accuracy by Transparency')
plt.ylim(0, 1.05)
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(x='Color', y='Accuracy', hue='Algorithm', data=df_plot, errorbar='se')
plt.title('Algorithm Accuracy by Color')
plt.ylim(0, 1.05)
plt.show()

# Short-window curves: feasibility assessment for real-time control
window_rows = []
for window_s in [0.5, 1.0, 2.0, 3.0, DECISION_WINDOW_S]:
    win_tmax = min(ANALYSIS_TMIN + window_s, ANALYSIS_TMAX)
    epochs_win = epochs.copy().crop(tmin=ANALYSIS_TMIN, tmax=win_tmax, include_tmax=False)
    for method in ['CCA', 'FBCCA']:
        df_win = evaluate_epochs(epochs_win, TARGET_FREQS, method=method)
        acc = df_win['Correct'].mean()
        window_rows.append({
            'Window_s': win_tmax - ANALYSIS_TMIN,
            'Algorithm': method,
            'Accuracy': acc,
            'ITR_bits_min': calculate_itr(len(TARGET_FREQS), acc, win_tmax - ANALYSIS_TMIN),
        })

df_window = pd.DataFrame(window_rows)
print('\n=== Short-Window Performance Curves (CCA/FBCCA) ===')
display(df_window)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.lineplot(data=df_window, x='Window_s', y='Accuracy', hue='Algorithm', marker='o', ax=axes[0])
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Short-window Accuracy')
axes[0].set_xlabel('Decision window (s)')

sns.lineplot(data=df_window, x='Window_s', y='ITR_bits_min', hue='Algorithm', marker='o', ax=axes[1])
axes[1].set_title('Short-window ITR')
axes[1].set_xlabel('Decision window (s)')
axes[1].set_ylabel('bits/min')
plt.tight_layout()
plt.show()


## 12. Margin Threshold Rejection Curve


In [ ]:
def build_margin_rejection_curve(df_comp, algorithms, threshold_steps=21):
    rows = []
    for algo in algorithms:
        margin_col = f'Margin_{algo}'
        acc_col = f'Acc_{algo}'
        margins = df_comp[margin_col].dropna()
        if margins.empty:
            continue
        thresholds = np.linspace(0, float(margins.max()), threshold_steps)
        for threshold in thresholds:
            accepted = df_comp[margin_col] >= threshold
            accepted_n = int(accepted.sum())
            rejected_n = int((~accepted).sum())
            rows.append({
                'Algorithm': algo,
                'Threshold': float(threshold),
                'Accepted': accepted_n,
                'Rejected': rejected_n,
                'Reject_Rate': rejected_n / len(df_comp),
                'Accepted_Accuracy': float(df_comp.loc[accepted, acc_col].mean()) if accepted_n else np.nan,
                'Overall_Effective_Accuracy': float(df_comp.loc[accepted, acc_col].sum() / len(df_comp)),
            })
    return pd.DataFrame(rows)


df_margin_reject = build_margin_rejection_curve(df_comp, ['CCA', 'FBCCA', 'TRCA'])
print('Margin threshold rejection curve')
display(df_margin_reject)

print('Minimum threshold for accepted accuracy >= 90% per algorithm')
operating_rows = []
for algo, group in df_margin_reject.groupby('Algorithm'):
    candidates = group[(group['Accepted'] > 0) & (group['Accepted_Accuracy'] >= 0.90)]
    if candidates.empty:
        operating_rows.append({
            'Algorithm': algo,
            'Threshold': np.nan,
            'Accepted': 0,
            'Reject_Rate': np.nan,
            'Accepted_Accuracy': np.nan,
        })
    else:
        best = candidates.sort_values(['Threshold', 'Reject_Rate']).iloc[0]
        operating_rows.append(
            best[['Algorithm', 'Threshold', 'Accepted', 'Reject_Rate', 'Accepted_Accuracy']].to_dict())
df_margin_operating = pd.DataFrame(operating_rows)
display(df_margin_operating)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.lineplot(data=df_margin_reject, x='Reject_Rate', y='Accepted_Accuracy', hue='Algorithm', marker='o', ax=axes[0])
axes[0].axhline(0.9, color='red', linestyle='--', alpha=0.6)
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Accepted Accuracy vs Reject Rate')

sns.lineplot(data=df_margin_reject, x='Threshold', y='Accepted', hue='Algorithm', marker='o', ax=axes[1])
axes[1].set_title('Accepted Trials by Margin Threshold')
plt.tight_layout()
plt.show()


## 13. Rest/Cue False Activation Rate


In [ ]:
def score_epoch_data(data, method, target_freqs, sfreq):
    method = method.upper()
    if method == 'CCA':
        scores = cca_analysis(data, target_freqs, sfreq)
    elif method == 'FBCCA':
        scores = fbcca_analysis(data, target_freqs, sfreq)
    else:
        raise ValueError(f'False activation scoring does not support {method}.')
    pred_freq = max(scores, key=scores.get)
    return pred_freq, float(scores[pred_freq]), score_margin(scores)


def make_rest_windows(raw_obj, all_events, event_dict, available_chs, window_s=DECISION_WINDOW_S):
    rest_codes = {name: code for name, code in event_dict.items() if name == 'rest_eyes_open'}
    if not rest_codes:
        return np.empty((0, len(available_chs), 0))

    rest_epochs = mne.Epochs(
        raw_obj,
        all_events,
        event_id=rest_codes,
        tmin=0,
        tmax=120,
        baseline=None,
        preload=True,
        verbose=False,
    )
    rest_epochs.pick_channels(available_chs)
    rest_data = rest_epochs.get_data()[0]
    sfreq = rest_epochs.info['sfreq']
    win_n = int(round(window_s * sfreq))
    n_windows = rest_data.shape[-1] // win_n
    return np.stack([rest_data[:, i * win_n:(i + 1) * win_n] for i in range(n_windows)])


def make_cue_windows(raw_obj, all_events, event_dict, available_chs):
    cue_codes = {name: code for name, code in event_dict.items() if name == 'cue'}
    if not cue_codes:
        return np.empty((0, len(available_chs), 0))
    cue_epochs = mne.Epochs(
        raw_obj,
        all_events,
        event_id=cue_codes,
        tmin=0,
        tmax=1.0,
        baseline=None,
        preload=True,
        verbose=False,
    )
    cue_epochs.pick_channels(available_chs)
    return cue_epochs.get_data()


def evaluate_false_activation(data_segments, segment_type, thresholds_by_algo):
    rows = []
    if len(data_segments) == 0:
        return rows
    sfreq = epochs.info['sfreq']
    for algo, thresholds in thresholds_by_algo.items():
        scores = [score_epoch_data(seg, algo, TARGET_FREQS, sfreq) for seg in data_segments]
        margins = np.array([x[2] for x in scores], dtype=float)
        for label, threshold in thresholds:
            activations = margins >= threshold
            rows.append({
                'Segment_Type': segment_type,
                'Algorithm': algo,
                'Threshold_Label': label,
                'Threshold': threshold,
                'Segments': len(data_segments),
                'False_Activations': int(activations.sum()),
                'False_Activation_Rate': float(activations.mean()),
                'Mean_Margin': float(np.nanmean(margins)),
                'P95_Margin': float(np.nanpercentile(margins, 95)),
            })
    return rows


thresholds_by_algo = {}
for algo in ['CCA', 'FBCCA']:
    thresholds = [('0', 0.0)]
    op = df_margin_operating[df_margin_operating['Algorithm'] == algo]
    if not op.empty and pd.notna(op.iloc[0]['Threshold']):
        thresholds.append(('accepted_acc>=90%', float(op.iloc[0]['Threshold'])))
    thresholds.append(('stim_margin_p75', float(df_comp[f'Margin_{algo}'].quantile(0.75))))
    thresholds_by_algo[algo] = thresholds

rest_segments = make_rest_windows(raw, all_events, event_dict, available_chs)
cue_segments = make_cue_windows(raw, all_events, event_dict, available_chs)

false_rows = []
false_rows.extend(evaluate_false_activation(rest_segments, 'rest_eyes_open_4s', thresholds_by_algo))
false_rows.extend(evaluate_false_activation(cue_segments, 'cue_1s', thresholds_by_algo))
df_false_activation = pd.DataFrame(false_rows)
print('Rest/Cue false activation rate; false activation = non-stimulus segment with margin >= threshold.')
display(df_false_activation)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=df_false_activation,
    x='Threshold_Label',
    y='False_Activation_Rate',
    hue='Segment_Type',
    errorbar=None,
)
plt.title('False Activation Rate on Rest/Cue Segments')
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()
